# Növénybetegségek felismerése neurális hálóval

Cuda 13.0

In [ ]:
%pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu130

In [16]:
import shutil
import kagglehub
import torch
from torchvision import datasets, transforms, models
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import os

## Az adathalmaz
Kaggle: [PlantVillage](https://www.kaggle.com/datasets/mohitsingh1804/plantvillage)

In [ ]:
shutil.rmtree("./data/")

local_path = "./data"
path = kagglehub.dataset_download("mohitsingh1804/plantvillage")

shutil.move(path, local_path)

print("Dataset downloaded and moved to:", local_path)

Batch méret és kép méret (ResNet miatt)

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

BATCH_SIZE = 32
IMG_SIZE = 224

train_dir = "data/PlantVillage/train"
val_dir = "data/PlantVillage/val"

Device: cuda


In [18]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet
        std=[0.229, 0.224, 0.225]
    )
])

Eredeti *val* halmaz -> *test* halmaz

*train* 20% -> új *val* halmaz

In [19]:
full_train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

test_dataset = datasets.ImageFolder(root=val_dir, transform=transform)

In [20]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
class_names = full_train_dataset.classes
num_classes = len(class_names)

print("Osztályok száma:", num_classes)

Osztályok száma: 38


In [22]:
images, labels = next(iter(train_loader))

print("Képek shape:", images.shape)   # [batch, 3, 224, 224]
print("Címkék shape:", labels.shape)

Képek shape: torch.Size([32, 3, 224, 224])
Címkék shape: torch.Size([32])
